# Ball-pushing metrics — a live reference

This notebook is the companion to
[`src/ballpushing_utils/README_Ballpushing_metrics.md`](../src/ballpushing_utils/README_Ballpushing_metrics.md).
The README is the authoritative spec; this notebook is the
**operational view** — every metric is grouped by theme, described in
one line, and paired with the live value computed from a real fly.

Use it to answer questions like:

- *"What exactly do I get when I write `fly.event_summaries`?"*
- *"Which threshold in `Config` controls `first_major_event`?"*
- *"Is `distance_moved` in pixels or millimetres?"*
- *"I only care about pauses and chamber time — what's the minimal
  metric set I can turn on?"*

If you haven't read
[`ballpushing_utils_walkthrough.ipynb`](./ballpushing_utils_walkthrough.ipynb)
yet, do that first — it covers how to construct a `Fly` and what
`event_summaries` actually is at the dict level.

## 0 · Setup

Same trick as the walkthrough notebook: load a real fly via
`BALLPUSHING_DATA_ROOT` (override with `BALLPUSHING_WALKTHROUGH_FLY`);
if the path isn't reachable, every data cell prints `(skipped)` and
the Markdown explanations still read fine.

In [1]:
from __future__ import annotations

import os
from pathlib import Path

import numpy as np
import pandas as pd

from ballpushing_utils import Config, Fly, data_root, load_dotenv

load_dotenv(Path.cwd().parent / ".env")
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 140)
pd.set_option("display.max_colwidth", 80)

DATA_ROOT = data_root()
FLY_PATH = Path(os.environ.get(
    "BALLPUSHING_WALKTHROUGH_FLY",
    DATA_ROOT
        / "Ballpushing_Exploration/Videos/230704_FeedingState_1_AM_Videos_Tracked"
        / "arena2" / "corridor5",
))

fly = None
summaries: dict = {}
ball_key: str | None = None
if FLY_PATH.is_dir():
    fly = Fly(FLY_PATH, as_individual=True)
    summaries = fly.event_summaries or {}
    ball_key = next(iter(summaries), None)
    print(f"Loaded: {fly.metadata.name}")
    print(f"Available ball keys: {list(summaries.keys())}")
else:
    print("⚠ Fly path not reachable — data cells will be skipped.")
    print(f"Tried: {FLY_PATH}")

⚠️  WARNING: No SLEAP track names found for 231130_TNT_Fine_2_Videos_Tracked_arena2_corridor5!


Loaded: 231130_TNT_Fine_2_Videos_Tracked_arena2_corridor5
Available ball keys: ['fly_0_ball_0']


### Helper — render a metric table

Every thematic section below calls `show_metrics(names, descriptions)`
to produce a three-column table: `metric / value / description`. The
value column reads straight from `fly.event_summaries[ball_key]`.

In [2]:
def show_metrics(catalog: dict[str, str]) -> pd.DataFrame:
    """Build a metric × (value, description) table for the loaded fly.

    Parameters
    ----------
    catalog : dict[metric_name, description]
        Ordered mapping; the table preserves insertion order.

    Returns a dataframe even when the fly is absent — the value column
    shows '(no fly loaded)' so you can still read the descriptions.
    """
    if not summaries:
        rows = [(k, "(no fly loaded)", d) for k, d in catalog.items()]
    else:
        s = summaries[ball_key]
        rows = []
        for name, desc in catalog.items():
            val = s.get(name, "(not computed)")
            if isinstance(val, float) and np.isfinite(val):
                val = round(val, 4)
            rows.append((name, val, desc))
    return pd.DataFrame(rows, columns=["metric", "value", "description"]).set_index("metric")

## 1 · Core event definitions

Ball-pushing metrics are computed on top of a handful of event
definitions. The thresholds that classify them live in `Config` and
can be overridden per-fly via `custom_config`:

| Event          | Default threshold (px)            | `Config` field                     |
| -------------- | --------------------------------- | ---------------------------------- |
| Interaction    | fly–ball distance ≤ 45 px (2.7 mm) | `interaction_threshold = (0, 70)` |
| Significant    | ball displacement > 5 px (0.3 mm) | `significant_threshold = 5`        |
| Major          | ball displacement ≥ 20 px (1.2 mm) | `major_event_threshold = 20`       |
| Success direction | ball displacement ≥ 25 px (1.5 mm) | `success_direction_threshold = 25` |
| Final (std.)   | ball displacement 170 px (10.2 mm) | `final_event_threshold = 170`      |
| Final (F1)     | ball displacement 100 px (6.0 mm) | `final_event_F1_threshold = 100`   |

Pixels convert to millimetres via `Config.pixels_per_mm = 500 / 30 ≈
16.67`. Tweak any of these thresholds and `event_summaries` recomputes
with the new classification.

In [3]:
cfg = Config()
pd.Series({
    "interaction_threshold": cfg.interaction_threshold,
    "significant_threshold": cfg.significant_threshold,
    "major_event_threshold": cfg.major_event_threshold,
    "success_direction_threshold": cfg.success_direction_threshold,
    "final_event_threshold": cfg.final_event_threshold,
    "final_event_F1_threshold": cfg.final_event_F1_threshold,
    "pixels_per_mm": cfg.pixels_per_mm,
}, name="default thresholds")

interaction_threshold            (0, 45)
significant_threshold                  5
major_event_threshold                 20
success_direction_threshold           25
final_event_threshold                170
final_event_F1_threshold             100
pixels_per_mm                  16.666667
Name: default thresholds, dtype: object

## 2 · Task performance

The "did the fly learn it and how fast?" metrics. These are the
headline numbers that feature in most paper figures.

In [4]:
show_metrics({
    "has_finished": "1 if the fly reached the final-event threshold, else 0.",
    "has_major": "1 if at least one major event (≥ 20 px push) exists, else 0.",
    "has_significant": "1 if ≥ 1 significant event (> 5 px) exists, else 0.",
    "max_distance": "Max ball displacement from start (pixels).",
    "raw_max_distance": "Max displacement before any time/cut-off filtering (pixels).",
    "max_event": "Event index (0-based) at which max_distance was achieved.",
    "max_event_time": "Wall-clock time (s) of the max_distance event.",
    "final_event": "Event index that first crossed the final-event threshold.",
    "final_event_time": "Wall-clock time (s) of the final event.",
    "first_major_event": "Index of the 'aha-moment' event (first ≥ 20 px push).",
    "first_major_event_time": "Wall-clock time (s) of the first major event.",
    "major_event_first": "True if the major event is also the first event overall.",
    "first_significant_event": "Index of the first significant event (> 5 px).",
    "first_significant_event_time": "Wall-clock time (s) of the first significant event.",
})

,value,description
metric,,
has_finished,1,"1 if the fly reached the final-event threshold, else 0."
has_major,1,"1 if at least one major event (≥ 20 px push) exists, else 0."
has_significant,1,"1 if ≥ 1 significant event (> 5 px) exists, else 0."
max_distance,204.1562,Max ball displacement from start (pixels).
raw_max_distance,200.1226,Max displacement before any time/cut-off filtering (pixels).
max_event,4,Event index (0-based) at which max_distance was achieved.
max_event_time,2751.4483,Wall-clock time (s) of the max_distance event.
final_event,4,Event index that first crossed the final-event threshold.
final_event_time,2751.4483,Wall-clock time (s) of the final event.


## 3 · Interaction behaviour

Rate, efficiency, and directional strategy of ball contacts.

In [5]:
show_metrics({
    "nb_events": "Count of interaction events, adjusted for time-to-exit chamber.",
    "nb_significant_events": "Count of events with > 5 px displacement.",
    "significant_ratio": "nb_significant_events / nb_events.",
    "overall_interaction_rate": "Events per second across the usable window.",
    "interaction_persistence": "Mean duration (s) of a single interaction event.",
    "interaction_proportion": "Fraction of accessible time spent interacting with the ball.",
    "cumulated_breaks_duration": "Total time (s) spent between interaction events (disengagement).",
    "time_to_first_interaction": "Latency (s) from recording start to the first interaction.",
    "pushed": "Count of significant events that moved the ball away from the start.",
    "pulled": "Count of significant events that moved the ball toward the start.",
    "pulling_ratio": "pulled / (pushed + pulled). 0.5 = balanced.",
    "success_direction": "'push' / 'pull' / 'both' — classifies the dominant strategy.",
})

,value,description
metric,,
nb_events,2.2287,"Count of interaction events, adjusted for time-to-exit chamber."
nb_significant_events,3,Count of events with > 5 px displacement.
significant_ratio,0.6,nb_significant_events / nb_events.
overall_interaction_rate,0.0022,Events per second across the usable window.
interaction_persistence,11.6759,Mean duration (s) of a single interaction event.
interaction_proportion,0.0211,Fraction of accessible time spent interacting with the ball.
cumulated_breaks_duration,0,Total time (s) spent between interaction events (disengagement).
time_to_first_interaction,3.1379,Latency (s) from recording start to the first interaction.
pushed,3,Count of significant events that moved the ball away from the start.


## 4 · Distance and ball displacement

How far and how efficiently the fly moved the ball. The `*_10min`
through `*_60min` variants freeze the denominator at a fixed time
window, which is useful for comparing flies of different durations.

In [6]:
show_metrics({
    "distance_moved": "Sum of per-event Euclidean displacements (pixels).",
    "raw_distance_moved": "distance_moved without final-event truncation.",
    "distance_ratio": "distance_moved / max_distance. Near 1.0 = directional, higher = back-and-forth.",
    "distance_moved_10min": "Ball-distance moved within the first 10 min.",
    "distance_moved_20min": "…within the first 20 min.",
    "distance_moved_30min": "…30 min.",
    "distance_moved_40min": "…40 min.",
    "distance_moved_50min": "…50 min.",
    "distance_moved_60min": "…60 min (full standard experiment).",
    "fly_distance_moved": "Total thorax-to-thorax path length traveled by the fly (mm).",
})

,value,description
metric,,
distance_moved,232.6960,Sum of per-event Euclidean displacements (pixels).
raw_distance_moved,1372.3386,distance_moved without final-event truncation.
distance_ratio,1.1398,"distance_moved / max_distance. Near 1.0 = directional, higher = back-and-forth."
distance_moved_10min,334.3310,Ball-distance moved within the first 10 min.
distance_moved_20min,334.3310,…within the first 20 min.
distance_moved_30min,334.3310,…30 min.
distance_moved_40min,334.3310,…40 min.
distance_moved_50min,334.3310,…50 min.
distance_moved_60min,334.3310,…60 min (full standard experiment).


## 5 · Spatial / chamber behaviour

When does the fly leave the start chamber? How much time does it
spend there? These metrics are sensitive to exploration vs freeze
phenotypes.

In [7]:
show_metrics({
    "chamber_exit_time": "Time (s) at which the fly permanently leaves the start chamber.",
    "chamber_time": "Total time (s) spent within chamber_radius of the start.",
    "chamber_ratio": "chamber_time / experiment duration. 1.0 = never left; 0.0 = never inside.",
    "time_chamber_beginning": "Time (s) spent in the chamber during the first 25% of the recording.",
    "persistence_at_end": "Fraction of frames with the fly at/near the corridor end (> 170 px from start).",
    "fraction_not_facing_ball": "Fraction of frames with body-axis > 30° off the fly→ball line.",
    "f1_exit_time": "F1 only: time (s) at which the fly exits the training corridor into the test one.",
})

,value,description
metric,,
chamber_exit_time,10.5517,Time (s) at which the fly permanently leaves the start chamber.
chamber_time,2530.0690,Total time (s) spent within chamber_radius of the start.
chamber_ratio,0.9195,chamber_time / experiment duration. 1.0 = never left; 0.0 = never inside.
time_chamber_beginning,874.2414,Time (s) spent in the chamber during the first 25% of the recording.
persistence_at_end,0.0973,Fraction of frames with the fly at/near the corridor end (> 170 px from start).
fraction_not_facing_ball,0.0568,Fraction of frames with body-axis > 30° off the fly→ball line.
f1_exit_time,NaN,F1 only: time (s) at which the fly exits the training corridor into the test...


## 6 · Locomotor activity — speed and pauses

`Config.keep_idle` and pause thresholds gate this section.
Pauses are bucketed by duration so you can tell a brief hesitation
from a full behavioural arrest.

In [8]:
show_metrics({
    "normalized_speed": "Mean fly speed / ball distance (dimensionless).",
    "speed_during_interactions": "Mean fly speed while in contact with the ball (px/s).",
    "speed_trend": "Linear slope of speed over time (px/s²). + = accelerating.",
    "overall_slope": "Linear slope of ball distance vs time across the whole session.",
    "auc": "Area under the cumulative ball-distance curve (trapezoidal).",
    "nb_stops": "Count of brief pauses (2-5 s).",
    "total_stop_duration": "Total time in brief pauses (s).",
    "median_stop_duration": "Median brief-pause duration (s).",
    "nb_pauses": "Count of medium pauses (5-10 s).",
    "total_pause_duration": "Total time in medium pauses (s).",
    "median_pause_duration": "Median medium-pause duration (s).",
    "nb_long_pauses": "Count of long pauses (≥ 10 s).",
    "total_long_pause_duration": "Total time in long pauses (s).",
    "median_long_pause_duration": "Median long-pause duration (s).",
    "has_long_pauses": "1 if any long pause (≥ 10 s) exists.",
})

,value,description
metric,,
normalized_speed,4.3662,Mean fly speed / ball distance (dimensionless).
speed_during_interactions,4.2994,Mean fly speed while in contact with the ball (px/s).
speed_trend,0.0034,Linear slope of speed over time (px/s²). + = accelerating.
overall_slope,-0.0343,Linear slope of ball distance vs time across the whole session.
auc,NaN,Area under the cumulative ball-distance curve (trapezoidal).
nb_stops,0.0000,Count of brief pauses (2-5 s).
total_stop_duration,0.0000,Total time in brief pauses (s).
median_stop_duration,NaN,Median brief-pause duration (s).
nb_pauses,0.0000,Count of medium pauses (5-10 s).


## 7 · Skeleton-derived metrics

These require the `*_full_body.h5` skeleton track to be present next
to the fly/ball HDF5s. They describe *how* the fly manipulates the
ball — head-push vs leg-push, coordinated vs flailing.

In [9]:
show_metrics({
    "flailing": "Mean squared-speed motion energy across front-leg keypoints during contacts.",
    "head_pushing_ratio": "Fraction of contact frames where head is closer to ball than legs.",
    "leg_visibility_ratio": "Weighted front-leg visibility during contacts (0 = none, 1 = both).",
})

,value,description
metric,,
flailing,2.8019,Mean squared-speed motion energy across front-leg keypoints during contacts.
head_pushing_ratio,0.8852,Fraction of contact frames where head is closer to ball than legs.
leg_visibility_ratio,1.0000,"Weighted front-leg visibility during contacts (0 = none, 1 = both)."


## 8 · Learning-curve fits and binned trends

These metrics fit a model to the cumulative ball-distance curve. The
linear `learning_slope` is the obvious starting point; `logistic_*`
fits a sigmoid and returns its three parameters (`L`, `k`, `t0`) plus
the goodness-of-fit. Binned variants split the session into N fixed
windows so you can see where the activity concentrated.

In [10]:
show_metrics({
    "learning_slope": "Linear regression slope of cumulative distance vs time.",
    "learning_slope_r2": "R² of the linear fit.",
    "logistic_L": "Sigmoid asymptote (maximum achievable distance).",
    "logistic_k": "Sigmoid steepness — how fast the fly ramps up.",
    "logistic_t0": "Sigmoid midpoint (time at which growth is fastest).",
    "logistic_r2": "R² of the sigmoid fit.",
})

# Binned families are long; render just the first few entries.
if summaries:
    binned = {k: v for k, v in summaries[ball_key].items()
              if k.startswith(("binned_slope_", "interaction_rate_bin_", "binned_auc_"))}
    if binned:
        print("\nBinned-metric families (first 10 entries):")
        display(pd.Series(binned).head(10).to_frame("value"))
    else:
        print("No binned_* metrics in this fly's summary (check enabled_metrics).")


Binned-metric families (first 10 entries):


,value
binned_slope_0,-0.116585
binned_slope_1,-0.000733
binned_slope_2,-0.000675
binned_slope_3,-0.000853
binned_slope_4,0.000034
binned_slope_5,-0.000649
binned_slope_6,0.007974
binned_slope_7,-0.266583
binned_slope_8,0.008527
binned_slope_9,-0.111552


## 9 · Everything else — what's actually in this fly's summary?

The tables above cover the documented metrics. Whatever you see here
and not above is either an F1-only metric, an experiment-specific
addon, or a new entry that hasn't made it to the README yet. If a
metric appears here that you think deserves documentation, add it to
`README_Ballpushing_metrics.md` and to this notebook.

In [11]:
if not summaries:
    print("(skipped — no fly loaded)")
else:
    all_keys = set(summaries[ball_key].keys())
    documented_above = {
        "has_finished", "has_major", "has_significant",
        "max_distance", "raw_max_distance", "max_event", "max_event_time",
        "final_event", "final_event_time",
        "first_major_event", "first_major_event_time", "major_event_first",
        "first_significant_event", "first_significant_event_time",
        "nb_events", "nb_significant_events", "significant_ratio",
        "overall_interaction_rate", "interaction_persistence", "interaction_proportion",
        "cumulated_breaks_duration", "time_to_first_interaction",
        "pushed", "pulled", "pulling_ratio", "success_direction",
        "distance_moved", "raw_distance_moved", "distance_ratio", "fly_distance_moved",
        "distance_moved_10min", "distance_moved_20min", "distance_moved_30min",
        "distance_moved_40min", "distance_moved_50min", "distance_moved_60min",
        "chamber_exit_time", "chamber_time", "chamber_ratio", "time_chamber_beginning",
        "persistence_at_end", "fraction_not_facing_ball", "f1_exit_time",
        "normalized_speed", "speed_during_interactions", "speed_trend",
        "overall_slope", "auc",
        "nb_stops", "total_stop_duration", "median_stop_duration",
        "nb_pauses", "total_pause_duration", "median_pause_duration",
        "nb_long_pauses", "total_long_pause_duration", "median_long_pause_duration",
        "has_long_pauses",
        "flailing", "head_pushing_ratio", "leg_visibility_ratio",
        "learning_slope", "learning_slope_r2",
        "logistic_L", "logistic_k", "logistic_t0", "logistic_r2",
        "fly_idx", "ball_idx", "ball_identity",
    }
    # Exclude binned families (covered as a group in § 8)
    undocumented = sorted(
        k for k in all_keys - documented_above
        if not k.startswith(("binned_slope_", "interaction_rate_bin_", "binned_auc_"))
    )
    if undocumented:
        print(f"{len(undocumented)} undocumented key(s) in this fly's summary:")
        display(pd.Series({k: summaries[ball_key][k] for k in undocumented}, name="value").to_frame())
    else:
        print("Every key in this fly's summary is documented above. Nice.")

4 undocumented key(s) in this fly's summary:


,value
ball_proximity_proportion_140px,NaN
ball_proximity_proportion_200px,NaN
ball_proximity_proportion_70px,NaN
raw_pauses,"[(1834.0689655172414, 1835.0689655172414, 1.0), (2128.551724137931, 2129.620..."


## 10 · Turning metrics on and off

Computing every metric is expensive for a full experiment. `Config`
exposes an `enabled_metrics` list that gates the expensive ones:

```python
from ballpushing_utils.config import _default_enabled_metrics
cfg = Config()
cfg.enabled_metrics = _default_enabled_metrics()     # paper defaults
cfg.enabled_metrics = None                           # compute everything
cfg.enabled_metrics = ["nb_events", "chamber_time"]  # minimal
```

The enabled set skips binned slopes, R² values, logistic fits, and
other low-signal derivatives. If you're iterating fast, start with
the minimal list.

When you hit a metric that's missing from a fly's summary, check
`fly.config.enabled_metrics` first — silent skipping via
`enabled_metrics` is the usual culprit.

## Where to go next

- Per-event view (one row per interaction rather than one row per
  fly) lives in `fly.event_metrics` — see
  `Dataset(..., dataset_type="event_metrics")` in
  [`dataset_types_guide.ipynb`](./dataset_types_guide.ipynb).
- F1-specific metrics (training-ball vs test-ball split,
  direction_match, path_efficiency_to_test) are in
  `fly.f1_metrics`.
- Learning-paradigm metrics (trial detection, per-trial durations)
  are in `fly.learning_metrics`.
- To spot events with timestamps for video scrubbing, use the
  diagnostics notebook + Panel dashboard:
  `notebooks/diagnostics_demo.ipynb` and
  `tools/diagnostics_dashboard.py`.